# 04 · Machine learning (scaffold)
Chronological splits, leakage-safe lag features, baselines vs ML. Remote access and preprocessing were validated on 2026-10-30; the series has ~400 rows, so training takes seconds. **Model selection uses the validation period only; touch the test period once, at the end.**

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
from src import utils, data_access, preprocessing, visualisation
settings = utils.load_settings()
utils.set_seed(settings['project']['random_seed'])
%matplotlib inline
from src import modelling

In [ ]:
RUN_TRAINING = True
EVALUATE_TEST = False   # set True only for the final, one-off evaluation

In [ ]:
import pandas as pd
s = pd.read_csv('../data/processed/thetao_study_area_monthly.csv', index_col=0, parse_dates=True).iloc[:, 0]
df = modelling.make_features(s, settings['modelling']['lags'])
m = settings['modelling']
split = modelling.chronological_split(df, m['train_end'], m['validation_end'], m['test_end'])
{k: (len(v), str(v.index.min().date()), str(v.index.max().date()))
 for k, v in vars(split).items()}

In [ ]:
if RUN_TRAINING:
    results = modelling.evaluate(split, settings['project']['random_seed'], on='validation')
    results.round(3).to_csv('../outputs/tables/04_validation_metrics.csv')
    display(results.round(3))

Skill > 0 means better than the baseline. Seasonal-cycle–dominated targets make climatology a strong baseline; beating it, not persistence, is the meaningful test.

In [ ]:
if RUN_TRAINING and EVALUATE_TEST:
    display(modelling.evaluate(split, settings['project']['random_seed'], on='test').round(3))